In [13]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [54]:
sentiment = pd.read_csv("../data/processed/reviews.csv")
sentiment.head()

,review_id,text,text_clean,rating,review_date,sentiment,issues,summary
0,75121e2c-dde8-4ec6-9a40-2cb735e5bec7,aplikasinya rekomen banget buat pemula yang ma...,aplikasinya rekomen banget buat pemula yang ma...,5,2026-04-13,positive,[],"Recommended app for beginners, easy to use for..."
1,8de6e3a5-39c2-4b1b-8686-ff0831d9a173,mau login selalu muncul konfigurasi.. tah pape...,mau login selalu muncul konfigurasi.. tah pape...,1,2026-04-13,negative,['login configuration error'],"Login always shows configuration error, unclea..."
2,bd6dd8a5-b6ad-4b6d-be71-154b5ef45dcd,"keranjang gak berfungsi, tiap d masukkan keran...","keranjang gak berfungsi, tiap d masukkan keran...",2,2026-04-13,negative,"['cart not working', 'items disappear from car...",Cart functionality is broken; items disappear ...
3,c952219d-bf42-42b6-bc7d-898521cddf10,pembelian pulsa sering error.. pulsa tidak mas...,pembelian pulsa sering error.. pulsa tidak mas...,1,2026-04-11,negative,"['purchase error', 'pulsa not received', 'bala...",Pulsa purchase often fails; balance deducted b...
4,edc9233a-c801-4810-bfab-f96c80f2bfe0,⭐⭐⭐⭐⭐,⭐⭐⭐⭐⭐,5,2026-04-11,positive,[],Five-star rating.


In [3]:
df = sentiment.copy()

In [24]:
df["Month"] = pd.to_datetime(df["review_date"]).dt.to_period("M")

trend = df.groupby("Month")["sentiment"].value_counts(normalize=False).unstack()

trend.index = trend.index.astype(str)
trend["total"] = trend[["positive", "neutral", "negative"]].sum(axis=1)

trend_pct = trend[["positive", "neutral", "negative"]].div(trend["total"], axis=0) * 100
trend["month"] = pd.to_datetime(trend.index).strftime("%b")

In [69]:
import plotly.express as px

rating_counts = df["rating"].value_counts().sort_index()

fig = px.bar(
    x=rating_counts.index,
    y=rating_counts.values,
    title="<b>2026 Pospay's Google Play Rating Distribution</b>",
    text=rating_counts.values
)

fig.update_traces(
    width=0.8  # 👈 CONTROL WIDTH HERE
)

fig.update_layout(
    xaxis_title="Rating (Stars)",
    yaxis_title="Number of Reviews",
    template="plotly_white"
)

fig.show()

In [70]:
fig = px.histogram(df, x="sentiment", title="<b>2026 Pospay's Google Play Sentiment Distribution</b>", text_auto=True)
#edit xaxis and yaxis title
fig.update_layout(
    xaxis_title="Sentiment",
    yaxis_title="Number of Reviews",
    template="plotly_white"
)
fig.show()

In [48]:
import pandas as pd
import plotly.graph_objects as go

# ===== PREP DATA =====
trend.index = pd.to_datetime(trend.index)
trend["month"] = trend.index.strftime("%b")

trend["total"] = trend[["positive", "neutral", "negative"]].sum(axis=1)

trend_pct = trend[["positive", "neutral", "negative"]].div(trend["total"], axis=0) * 100


# ===== BUILD FIGURE =====
fig = go.Figure()

# 🔹 GROUPED BARS (VISIBLE)
for s in ["negative", "positive", "neutral"]:
    fig.add_bar(
        x=trend["month"],
        y=trend[s],
        name=s.capitalize(),
        marker=dict(color=colors[s], opacity=0.4),

        text=[f"{v:.1f}%" for v in trend_pct[s]],

        textposition="inside",

        textfont=dict(size=13, color="white")
    )

line_colors = {"negative": "red", "positive": "green", "neutral": "darkblue"}


for s in ["negative", "positive", "neutral"]:
    fig.add_scatter(
        x=trend["month"],
        y=trend[s],
        mode="lines+markers",
        name=f"{s.capitalize()} Trend",
        line=dict(color=line_colors[s], width=3)
    )


# 🔥 TOTAL (FIXED POSITION)
fig.add_scatter(
    x=trend["month"],
    y=trend["negative"] * 1.1,
    mode="text",
    text=[f"<b>Total Review: {v}</b>" for v in trend["total"]],
    textposition="top center",
    showlegend=False
)

# ===== LAYOUT =====
fig.update_layout(
    barmode="overlay",  # 👈 FIXED
    title="<b>Trend Review per Bulan</b>",
    xaxis_title="Month",
    yaxis_title="Count",
    template="plotly_white"
)

fig.show()

In [72]:
from collections import Counter

negative_reviews = df[df["sentiment"] == "negative"]
negative_issue_counts = Counter(negative_reviews["issues"].explode())

top_issues = negative_issue_counts.most_common(10)
top_issues

[("['frequent errors']", 2),
 ("['login configuration error']", 1),
 ("['cart not working', 'items disappear from cart', 'inconvenient to retype items']",
  1),
 ("['purchase error', 'pulsa not received', 'balance deducted incorrectly']",
  1),
 ("['frequent password changes required', 'frequent phone number changes required', 'inconvenient login process']",
  1),
 ("['account blocked', 'cannot login']", 1),
 ("['incorrect password error despite correct input', 'unclear error messages']",
  1),
 ("['app cannot open', 'long update time']", 1),
 ("['account balance lost', 'unclear policy on balance expiration']", 1),
 ("['complicated purchase process', 'forced e-banking connection', 'poor user experience']",
  1)]

In [74]:
negative_reviews["issues"].explode().value_counts().head(20)

issues
['frequent errors']                                                                                               2
['login configuration error']                                                                                     1
['cart not working', 'items disappear from cart', 'inconvenient to retype items']                                 1
['purchase error', 'pulsa not received', 'balance deducted incorrectly']                                          1
['frequent password changes required', 'frequent phone number changes required', 'inconvenient login process']    1
['account blocked', 'cannot login']                                                                               1
['incorrect password error despite correct input', 'unclear error messages']                                      1
['app cannot open', 'long update time']                                                                           1
['account balance lost', 'unclear policy on balance expiration'] 

In [ ]:
CATEGORIES = [
    "authentication",
    "transaction",
    "security",       # NEW
    "feature",        # NEW
    "performance",
    "ux/ui",
    "support",
    "pricing",        # NEW
    "other"
]

def map_issue(issue):
    issue = issue.lower()

    # 🔴 Authentication
    if any(k in issue for k in [
        "login", "password", "access", "verification", "email", "face", "akun", "validasi", "upgrade", "PIN", "username", "credentials"
    ]):
        return "authentication"

    # 🔴 Security (NEW)
    if any(k in issue for k in [
        "fraud", "unauthorized", "tidak sah", "keamanan", "uang berkurang", "account ballance", "phishing", "scam", "hacker", "breach"
    ]):
        return "security"

    # 🔴 Transaction
    if any(k in issue for k in [
        "transfer", "balance", "pulsa", "topup", "payment", "transaksi", "refund", "withdrawal", "deposit", "billing", "invoice", "bank transfer", "balance", "pulsa", "topup"
    ]):
        return "transaction"

    # 🔴 Feature availability (NEW)
    if any(k in issue for k in [
        "cannot", "tidak bisa", "tidak muncul", "not available", "fitur", "menu", "tokopedia", "section", "option"
    ]):
        return "feature"

    # 🟠 Performance
    if any(k in issue for k in [
        "slow", "lemot", "loading", "unresponsive", "always problematic", "frequent", "gangguan", "disruption", "server", "error", "crash", "hang", "lag", "timeout"
    ]):
        return "performance"

    # 🟠 UX/UI
    if any(k in issue for k in [
        "difficult", "complicated", "ribet", "not user-friendly", "ui", "appearance", "branding", "design", "navigasi", "interface"
    ]):
        return "ux/ui"


    # 🟡 Support
    if any(k in issue for k in [
        "support", "service", "customer", "help", "cs", "contact", "response", "respon", "tanggapan"
    ]):
        return "support"

    # 🟡 Pricing
    if any(k in issue for k in [
        "biaya", "fee", "expensive", "topup", "charge", "harga", "price", "topup"
    ]):
        return "pricing"

    return "other"

df_exploded = negative_reviews.explode("issues")
df_exploded["category"] = df_exploded["issues"].apply(map_issue)

df_exploded["category"].value_counts()

category
authentication    34
performance       17
feature           15
transaction       13
other             12
ux/ui              7
security           4
support            2
pricing            1
Name: count, dtype: int64

In [105]:
import pandas as pd

# =========================
# 1. EXPLODE (if not already)
# =========================
# df_exploded = negative_reviews.explode("issues")
# df_exploded["category"] = df_exploded["issues"].apply(map_issue)


# =========================
# 2. COUNT PER CATEGORY
# =========================
category_counts = (
    df_exploded["category"]
    .value_counts()
    .reset_index()
)
category_counts.columns = ["category", "count"]


# =========================
# 3. GET TOP ISSUES PER CATEGORY
# =========================
example_df = (
    df_exploded.groupby(["category", "issues"])
    .size()
    .reset_index(name="freq")
    .sort_values(["category", "freq"], ascending=[True, False])
)


# =========================
# 4. TAKE TOP 3 EXAMPLES
# =========================
top_examples = (
    example_df.groupby("category")
    .head(2)
    .groupby("category")["issues"]
    .apply(lambda x: "; ".join(x))
    .reset_index()
    .rename(columns={"issues": "examples"})
)


# =========================
# 5. MERGE COUNT + EXAMPLES
# =========================
summary_df = pd.merge(category_counts, top_examples, on="category")

summary_df = summary_df[["category", "examples", "count"]]

# Optional: sort by importance
summary_df = summary_df.sort_values("count", ascending=False)


# =========================
# 6. ADD PERCENTAGE (OPTIONAL)
# =========================
summary_df["percentage"] = (
    summary_df["count"] / summary_df["count"].sum() * 100
).round(1)



summary_df.to_csv("../data/processed/issue_summary.csv", index=False)

In [ ]:
summary_d

,category,example,count
0,authentication,"['OTP required every login', 'update caused in...",41
1,performance,['frequent errors'],16
2,feature,['Pos Pay no longer connected to Tokopedia'],15
3,other,"['aplikasi buruk', 'hanya janji tanpa realisasi']",10
4,transaction,['bank transfer money not received'],10
5,ux/ui,['aplikasi dianggap ribet'],7
6,security,"['app errors', 'suspected fraud']",4
7,support,['unprofessional service'],1
8,pricing,['biaya potongan terlalu tinggi'],1
